In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import xgboost as xgb

In [ ]:
# Set plot style
sns.set_style('whitegrid')

In [ ]:
# Load the dataset
df = pd.read_csv('./data/Rockfall.csv')

In [ ]:
# Remove leaky features
leaky_features = ['Susceptibility_Score', 'Impact_Level']
df_no_leak = df.drop(columns=leaky_features)

In [ ]:
# One-hot encode categorical variables
df_encoded = pd.get_dummies(df_no_leak, drop_first=True)

In [ ]:
# Feature and target selection (all features except target)
X = df_encoded.drop('Occurred', axis=1)
y = df_encoded['Occurred']

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

In [ ]:
# Data Balancing with SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print('Original training set shape:', X_train.shape)
print('Balanced training set shape:', X_train_bal.shape)

In [ ]:
# Random Forest on Balanced Data
print("\nRandom Forest (Balanced Data)")
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1,max_depth=20,min_samples_split=5,min_samples_leaf=1)
rf_model.fit(X_train_bal, y_train_bal)
y_pred_rf = rf_model.predict(X_test)
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Logistic Regression on Balanced Data
print("\nLogistic Regression (Balanced Data)")
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_bal, y_train_bal)
y_pred_logreg = logreg.predict(X_test)
print(classification_report(y_test, y_pred_logreg))

In [ ]:
# XGBoost Classifier on Balanced Data
print("\nXGBoost Classifier (Balanced Data)")
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_bal, y_train_bal)
y_pred_xgb = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

In [ ]:
# (Optional) Confusion Matrix for Best Model
cm = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Rockfall', 'Rockfall'], yticklabels=['No Rockfall', 'Rockfall'])
plt.title('Confusion Matrix (XGBoost)', fontsize=16)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.show()